# MiniGrid Benchmark Runner
Notebook para executar benchmark_minigrid em ambiente local, Colab ou Kaggle.

In [ ]:
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"Execution environment: {EXECUTION_ENV}")

In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.1.0
    !pip install -q langchain-openai>=0.0.1
    !pip install -q langchain-deepseek>=0.0.1
    !pip install -q langchain-huggingface==1.2.2
    !pip install -q transformers>=5.5.4

In [ ]:
import sys
import os

if EXECUTION_ENV == "colab":
    repo_path = os.path.join(os.getcwd(), "minigrid_benchmark")
elif EXECUTION_ENV == "kaggle":
    repo_path = "/kaggle/working/minigrid_benchmark"
else:
    cwd = os.getcwd()
    candidates = [cwd, os.path.abspath(os.path.join(cwd, ".."))]
    repo_path = None
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, "src", "benchmark_minigrid.py")):
            repo_path = candidate
            break
    if repo_path is None:
        repo_path = cwd

if EXECUTION_ENV in ["colab", "kaggle"] and not os.path.exists(repo_path):
    !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git {repo_path}

src_path = os.path.join(repo_path, "src")
sys.path.append(src_path)

print(f"repo_path={repo_path}")
print(f"src_path={src_path}")

In [ ]:
import experiments_util
from benchmark_minigrid import benchmark_minigrid

In [ ]:
if EXECUTION_ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    results_dir = '/content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results'
elif EXECUTION_ENV == "kaggle":
    results_dir = '/kaggle/working/results'
else:
    results_dir = os.path.abspath(os.path.join(repo_path, 'results'))

os.makedirs(results_dir, exist_ok=True)
experiments_util.RESULTS_BASE_DIR = results_dir
print(f"results_dir={results_dir}")

In [ ]:
#PROVIDER = "openai"  # openai | deepseek | huggingface
#MODEL_ID = "gpt-5.4-mini"
PROVIDER = "deepseek"
MODEL_ID = "deepseek-v4-flash"
QUANTIZATION = None

#PROVIDER = "hf"
#MODEL_ID = "WeiboAI/VibeThinker-3B"
#QUANTIZATION = "8bit"  # "8bit" or "4bit" or None

In [ ]:
API_KEY = None

if PROVIDER == "openai":
    secret_name = "OPENAI_PABLO" 
elif PROVIDER == "deepseek":
    secret_name = "DEEPSEEK_API_KEY"
elif PROVIDER == "huggingface":
    secret_name = "HF_TOKEN"
else:
    print("Invalid provider: ", PROVIDER)

if EXECUTION_ENV == "colab":
    from google.colab import userdata
    API_KEY = userdata.get(secret_name)
elif EXECUTION_ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret(secret_name)
else:
    API_KEY = os.getenv(secret_name)

In [ ]:
EXPERIMENT_NAME = f"benchmark_{PROVIDER}_{MODEL_ID}"

print(f"experiment_name={EXPERIMENT_NAME}")

In [ ]:
final_results, summary_path = benchmark_minigrid(
    provider=PROVIDER,
    model_id=MODEL_ID,
    experiment_name=EXPERIMENT_NAME,
    results_base_dir=results_dir,
    api_key=API_KEY,
    quantization=QUANTIZATION if PROVIDER == "hf" else None,
    verbose=True,
)

print('Saved summary to:', summary_path)

In [ ]:
final_results